## Questão 7 - Sistema de recomendação

### Cenário

A Marina percebeu que clientes que compram lanchas quase sempre esquecem de levar a defensa (proteção lateral). Ela quer implementar uma vitrine de "Quem comprou isso, também levou..." no site. 
Como não temos ferramentas de Big Data caras, você precisará criar um motor de recomendação, baseado na similaridade de compra dos clientes. 
Identificar qual produto deve ser recomendado junto ao item “Motor de Popa 1949”, com base na similaridade de comportamento de compra dos clientes.

### Tarefa

1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:

   a. Linhas: `customer_id`.

   b. Colunas: `product_id`.

   c. Valor da célula:

      - `1`, se o cliente comprou o produto ao menos uma vez;
      - `0`, caso contrário.

   d. A quantidade comprada deve ser ignorada; será considerada apenas a presença ou ausência da compra.

2. Calcule a similaridade entre produtos:

   a. Calcule a Similaridade de Cosseno (*Cosine Similarity*) entre os vetores dos produtos.

   b. A similaridade deve ser calculada no formato produto × produto, com base nos clientes que compraram cada item.

3. Gere o ranking de produtos similares:

   a. Considere o produto `Motor de Popa 1949` como item de referência.

   b. Gere um ranking com os nomes dos cinco produtos mais similares.

   c. Desconsidere o próprio `Motor de Popa 1949` do ranking.


## Questão 7.1 - Código em python
### Script em python que:
- Constrói a matriz usuário–item
- Calcula a similaridade de cosseno
- Gera o ranking de similaridade
- Bibliotecas permitidas:
     - pandas
     - numpy
     - sklearn (opcional, para cosine similarity)

In [ ]:
"""
Projeto: Desafio Lighthouse
Questão 7 - Sistema de recomendação por similaridade de cosseno

Objetivo:
Encontrar os produtos mais similares ao Motor de Popa 1949 com base
no comportamento de compra dos clientes.
"""

import pandas as pd


# ============================================================
# ETAPA 1 - Leitura dos datasets necessários
# ============================================================

# Carrega os pedidos para identificar quem realizou cada compra.
orders = pd.read_csv("orders.csv")

# Carrega os itens de cada pedido.
order_items = pd.read_csv("order_items.csv")

# Carrega a relação entre cada variação e seu produto principal.
product_variants = pd.read_csv("product_variants.csv")

# Carrega os nomes dos produtos.
products = pd.read_csv("products.csv")


# ============================================================
# ETAPA 2 - Criação da base cliente-produto
# Cadeia: orders → order_items → product_variants → products
# ============================================================

itens_clientes = (
    # Mantém a identificação do pedido e da variação comprada.
    order_items[["order_id", "product_variant_id"]]

    # Associa cada pedido ao cliente que o realizou.
    .merge(
        orders[["id", "customer_id"]].rename(columns={"id": "order_id"}),
        on="order_id",
        how="inner"
    )

    # Converte a variação comprada no produto correspondente.
    .merge(
        product_variants[["id", "product_id"]].rename(
            columns={"id": "product_variant_id"}
        ),
        on="product_variant_id",
        how="inner"
    )

    # Adiciona o nome do produto para facilitar a leitura do resultado.
    .merge(
        products[["id", "name"]].rename(
            columns={"id": "product_id", "name": "product_name"}
        ),
        on="product_id",
        how="inner"
    )
)

print("Base cliente-produto:")
display(itens_clientes.head())


# ============================================================
# ETAPA 3 - Remoção de compras repetidas do mesmo produto
# Cada combinação cliente-produto deve aparecer apenas uma vez.
# ============================================================

# A quantidade comprada e compras repetidas são ignoradas.
# É mantida apenas a informação de que o cliente comprou o produto.
interacoes = (
    itens_clientes[["customer_id", "product_id"]]
    .drop_duplicates()
)

print("\nInterações únicas cliente-produto:")
display(interacoes.head())


# ============================================================
# ETAPA 4 - Construção da matriz usuário-item
# Linhas: clientes | Colunas: produtos | Valores: 0 ou 1
# ============================================================

# crosstab cria uma tabela cruzada entre clientes e produtos.
# Inicialmente, cada célula representa a quantidade de interações.
matriz_usuario_item = pd.crosstab(
    interacoes["customer_id"],
    interacoes["product_id"]
)

# Converte qualquer valor maior que zero em 1.
# Assim, a matriz representa somente presença ou ausência de compra.
matriz_usuario_item = (
    matriz_usuario_item > 0
).astype(int)

print("\nMatriz usuário-item:")
display(matriz_usuario_item.iloc[:5, :5])


# ============================================================
# ETAPA 5 - Cálculo da similaridade produto × produto
# Fórmula:
# similaridade(A, B) = produto escalar / (norma de A × norma de B)
# ============================================================

# Transpõe a matriz:
# cada linha passa a representar um produto e cada coluna, um cliente.
vetores_produtos = matriz_usuario_item.T

# Calcula o produto escalar entre todos os pares de produtos.
# O resultado indica quantos clientes compraram os dois produtos.
produto_escalar = vetores_produtos.dot(vetores_produtos.T)

# Calcula a norma de cada vetor de produto.
# A norma é necessária para normalizar a comparação entre produtos
# com diferentes quantidades de compradores.
normas_produtos = (
    (vetores_produtos ** 2)
    .sum(axis=1)
    ** 0.5
)

# Divide o produto escalar pelas normas dos dois produtos.
# O resultado varia de 0 a 1:
# 0 = nenhum comportamento de compra compartilhado;
# 1 = comportamento de compra idêntico.
matriz_similaridade = (
    produto_escalar
    .div(normas_produtos, axis=0)
    .div(normas_produtos, axis=1)
)

print("\nMatriz de similaridade calculada.")

# Exibe uma amostra da matriz produto × produto.
# Linhas e colunas representam produtos; cada célula apresenta
# a similaridade de cosseno entre os respectivos produtos.
# Os valores foram arredondados para três casas somente na visualização.
print("\nAmostra da matriz de similaridade produto × produto:")
display(matriz_similaridade.round(3).iloc[:5, :5])

# ============================================================
# ETAPA 6 - Identificação do ID do produto de referência
# ============================================================

produto_referencia = "Motor de Popa 1949"

# Localiza o produto de referência pelo nome.
produto_alvo = products[
    products["name"] == produto_referencia
]

# Interrompe a execução caso o produto não exista na base.
if produto_alvo.empty:
    raise ValueError(
        f"O produto '{produto_referencia}' não foi encontrado."
    )

# Armazena o identificador do produto para localizar sua coluna
# na matriz de similaridade.
produto_alvo_id = produto_alvo["id"].iloc[0]


# ============================================================
# ETAPA 7 - Ranking dos cinco produtos mais similares
# O próprio produto de referência é removido do ranking.
# ============================================================

# Seleciona a similaridade entre o Motor de Popa 1949
# e todos os demais produtos.
ranking = (
    matriz_similaridade[produto_alvo_id]

    # Remove o próprio produto, cuja similaridade seria sempre igual a 1.
    .drop(produto_alvo_id)

    # Transforma a série em tabela para permitir ordenação e junção.
    .reset_index()
)

ranking.columns = ["product_id", "similaridade_cosseno"]

# Ordena os produtos da maior para a menor similaridade.
# Em caso de empate, utiliza o product_id como critério secundário.
ranking = (
    ranking
    .sort_values(
        by=["similaridade_cosseno", "product_id"],
        ascending=[False, True]
    )

    # Mantém somente os cinco produtos mais similares.
    .head(5)

    # Inclui o nome dos produtos recomendados.
    .merge(
        products[["id", "name"]].rename(
            columns={
                "id": "product_id",
                "name": "produto_recomendado"
            }
        ),
        on="product_id",
        how="left"
    )
)

# Arredonda a similaridade somente para apresentação.
ranking["similaridade_cosseno"] = (
    ranking["similaridade_cosseno"].round(4)
)

# Define as colunas finais apresentadas no ranking.
ranking = ranking[
    ["produto_recomendado", "similaridade_cosseno"]
]

print("\nTop 5 produtos mais similares ao Motor de Popa 1949:")
display(ranking)

Base cliente-produto:


,order_id,product_variant_id,customer_id,product_id,product_name
0,1,113,1136,59,Bateria Náutica 8789
1,2,293,618,146,Cabo Náutico 7323
2,2,366,618,180,Motor de Popa 1949
3,2,561,618,275,Cabo Náutico 5921
4,2,385,618,190,GPS Plotter 3107



Interações únicas cliente-produto:


,customer_id,product_id
0,1136,59
1,618,146
2,618,180
3,618,275
4,618,190



Matriz usuário-item:


product_id,1,2,3,4,5
customer_id,,,,,
1,0,1,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,1
5,0,0,1,0,0



Matriz de similaridade calculada.

Amostra da matriz de similaridade produto × produto:


product_id,1,2,3,4,5
product_id,,,,,
1,1.000,0.227,0.162,0.146,0.235
2,0.227,1.000,0.173,0.189,0.231
3,0.162,0.173,1.000,0.121,0.173
4,0.146,0.189,0.121,1.000,0.170
5,0.235,0.231,0.173,0.170,1.000



Top 5 produtos mais similares ao Motor de Popa 1949:


,produto_recomendado,similaridade_cosseno
0,Motor de Popa 5331,0.2566
1,Cabo Náutico 2105,0.2562
2,Vela Mestra 1913,0.2558
3,Cabo Náutico 9048,0.2393
4,GPS Plotter 6249,0.2377


In [4]:
"""
Projeto: Desafio Lighthouse
Questão 7 - Sistema de recomendação por similaridade de cosseno

Objetivo:
Encontrar os produtos mais similares ao Motor de Popa 1949 com base
no comportamento de compra dos clientes.
"""

import pandas as pd


# ============================================================
# ETAPA 1 - Leitura dos datasets necessários
# ============================================================
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
product_variants = pd.read_csv("product_variants.csv")
products = pd.read_csv("products.csv")


# ============================================================
# ETAPA 2 - Criação da base cliente-produto
# Cadeia: orders -> order_items -> product_variants -> products
# ============================================================
itens_clientes = (
    order_items[["order_id", "product_variant_id"]]
    .merge(
        orders[["id", "customer_id"]].rename(columns={"id": "order_id"}),
        on="order_id",
        how="inner"
    )
    .merge(
        product_variants[["id", "product_id"]].rename(
            columns={"id": "product_variant_id"}
        ),
        on="product_variant_id",
        how="inner"
    )
    .merge(
        products[["id", "name"]].rename(
            columns={"id": "product_id", "name": "product_name"}
        ),
        on="product_id",
        how="inner"
    )
)

print("Base cliente-produto:")
display(itens_clientes.head())


# ============================================================
# ETAPA 3 - Remoção de compras repetidas do mesmo produto
# Cada combinação cliente-produto deve aparecer apenas uma vez.
# ============================================================
interacoes = (
    itens_clientes[["customer_id", "product_id"]]
    .drop_duplicates()
)

print("\nInterações únicas cliente-produto:")
display(interacoes.head())


# ============================================================
# ETAPA 4 - Construção da matriz usuário-item
# Linhas: clientes | Colunas: produtos | Valores: 0 ou 1
# ============================================================
matriz_usuario_produto = pd.crosstab(
    interacoes["customer_id"],
    interacoes["product_id"]
)

matriz_usuario_produto = (
    matriz_usuario_produto > 0
).astype(int)

print("\nMatriz usuário-produto:")
display(matriz_usuario_produto.iloc[:5, :5])


# ============================================================
# ETAPA 5 - Cálculo de similaridade produto × produto
# Fórmula: cosseno(A, B) = produto_escalar / (norma_A × norma_B)
# ============================================================

# A transposição cria vetores de produtos compostos pelos clientes.
vetores_produtos = matriz_usuario_produto.T

# Produto escalar entre todos os pares de produtos.
produto_escalar = vetores_produtos.dot(vetores_produtos.T)

# Norma de cada vetor de produto.
normas_produtos = (vetores_produtos ** 2).sum(axis=1) ** 0.5

# Aplicação da fórmula da similaridade de cosseno.
matriz_similaridade = (
    produto_escalar
    .div(normas_produtos, axis=0)
    .div(normas_produtos, axis=1)
)

print("\nMatriz de similaridade calculada.")


# ============================================================
# ETAPA 6 - Identificação do ID do produto de referência
# ============================================================
produto_referencia = "Motor de Popa 1949"

produto_alvo = products[
    products["name"] == produto_referencia
]

if produto_alvo.empty:
    raise ValueError(f"O produto '{produto_referencia}' não foi encontrado.")

produto_alvo_id = produto_alvo["id"].iloc[0]


# ============================================================
# ETAPA 7 - Ranking dos cinco produtos mais similares
# O próprio produto de referência é removido do ranking.
# ============================================================
ranking = (
    matriz_similaridade[produto_alvo_id]
    .drop(produto_alvo_id)
    .reset_index()
)

ranking.columns = ["product_id", "similaridade_cosseno"]

ranking = (
    ranking
    .sort_values(
        by=["similaridade_cosseno", "product_id"],
        ascending=[False, True]
    )
    .head(5)
    .merge(
        products[["id", "name"]].rename(
            columns={"id": "product_id", "name": "produto_recomendado"}
        ),
        on="product_id",
        how="left"
    )
)

ranking["similaridade_cosseno"] = (
    ranking["similaridade_cosseno"].round(4)
)

ranking = ranking[
    ["produto_recomendado", "similaridade_cosseno"]
]

print("\nTop 5 produtos mais similares ao Motor de Popa 1949:")
display(ranking)

Base cliente-produto:


,order_id,product_variant_id,customer_id,product_id,product_name
0,1,113,1136,59,Bateria Náutica 8789
1,2,293,618,146,Cabo Náutico 7323
2,2,366,618,180,Motor de Popa 1949
3,2,561,618,275,Cabo Náutico 5921
4,2,385,618,190,GPS Plotter 3107



Interações únicas cliente-produto:


,customer_id,product_id
0,1136,59
1,618,146
2,618,180
3,618,275
4,618,190



Matriz usuário-produto:


product_id,1,2,3,4,5
customer_id,,,,,
1,0,1,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,1
5,0,0,1,0,0



Matriz de similaridade calculada.

Top 5 produtos mais similares ao Motor de Popa 1949:


,produto_recomendado,similaridade_cosseno
0,Motor de Popa 5331,0.2566
1,Cabo Náutico 2105,0.2562
2,Vela Mestra 1913,0.2558
3,Cabo Náutico 9048,0.2393
4,GPS Plotter 6249,0.2377


## Questão 7.3 - Explicação

### Como a matriz foi construída?

A matriz usuário-produto foi construída a partir dos relacionamentos entre `orders`, `order_items`, `product_variants` e `products`.

As linhas representam os clientes (`customer_id`) e as colunas representam os produtos (`product_id`). Cada célula recebeu valor `1` quando o cliente comprou aquele produto ao menos uma vez e valor `0` quando não houve compra.

Antes da criação da matriz, combinações repetidas de cliente e produto foram removidas. Dessa forma, a quantidade comprada não influenciou o resultado, conforme solicitado.

### O que significa a similaridade de cosseno nesse contexto?

A similaridade de cosseno compara os vetores de compra dos produtos. Cada produto é representado pelo conjunto de clientes que o compraram.

Uma similaridade mais alta indica que os produtos foram adquiridos por grupos de clientes mais parecidos. Portanto, se clientes que compraram o Motor de Popa 1949 também compraram frequentemente outro produto, esse produto recebe uma similaridade maior e pode ser recomendado.

O produto com maior similaridade ao Motor de Popa 1949 foi o **Motor de Popa 5331**, com similaridade de cosseno de **0,2566**.

### Limitação do método

O método considera somente presença ou ausência de compra. Informações como quantidade comprada, data da compra, preço, margem, disponibilidade em estoque e ordem de compra não são consideradas.

Além disso, produtos pouco comprados podem apresentar recomendações menos confiáveis por possuírem poucas interações. O método também pode favorecer produtos populares, mesmo quando não existe uma relação de complemento entre eles.
